In [ ]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path


#get element from name
def guess_element_from_name(atom_name):

    if isinstance(atom_name, bytes):
        atom_name = atom_name.decode("utf-8")

    atom_name = atom_name.strip()

    if atom_name.startswith("H"):
        return "H"
    elif atom_name.startswith("C"):
        return "C"
    elif atom_name.startswith("N"):
        return "N"
    elif atom_name.startswith("O"):
        return "O"
    elif atom_name.startswith("S"):
        return "S"
    else:
        return atom_name[0].upper()


#folder path
folder = (
    "/home/wilma/harryshouse/output_data/intensityvar_homogen"
)


#intensity directories
intensity_dirs = sorted(
    [
        d for d in os.listdir(folder)
        if d.startswith("I_")
    ],
    key=lambda x: float(x.replace("I_", ""))
)


atom_accumulator = {}


#loop over intensities
for intensity_dir in intensity_dirs:

    intensity_path = os.path.join(
        folder,
        intensity_dir
    )

    h5_files = sorted(
        Path(intensity_path).rglob("results.h5")
    )

    print(f"Processing {intensity_dir}")

    all_vels = []
    all_radii = []

    sulfur_info = None

    #loop over replicas
    for h5_file in h5_files:

        try:

            with h5py.File(h5_file, "r") as f:

                final_velocity = f["final_velocity"][:]
                initial_position = f["initial_position"][:]

                atom_names = f["atom_names"][:]
                residue_ids = f["residue_ids"][:]
                residue_names = f["residue_names"][:]

                

                atom_names = np.array([
                    x.decode("utf-8") if isinstance(x, bytes) else x
                    for x in atom_names
                ])

                residue_names = np.array([
                    x.decode("utf-8") if isinstance(x, bytes) else x
                    for x in residue_names
                ])

                #elements
                elements = np.array([
                    guess_element_from_name(a)
                    for a in atom_names
                ])

                sulfur_idx = np.where(elements == "S")[0]

                if len(sulfur_idx) == 0:
                    continue

                #store sulfur info

                if sulfur_info is None:

                    sulfur_info = {

                        "resids": residue_ids[sulfur_idx],
                        "resnames": residue_names[sulfur_idx]
                    }
                #protein center
                heavy_mask = elements != "H"

                center = np.mean(
                    initial_position[heavy_mask],
                    axis=0
                )

                #radius
                sulfur_coords = initial_position[sulfur_idx]

                distances = np.linalg.norm(
                    sulfur_coords - center[None, :],
                    axis=1
                )

              #velocities
                sulfur_velocities = final_velocity[sulfur_idx]

                all_vels.append(sulfur_velocities)
                all_radii.append(distances)

        except Exception as e:

            print(f"Error reading {h5_file}: {e}")

    
    if len(all_vels) < 2:
        continue


    V = np.array(all_vels)
    R = np.array(all_radii)

   #normalise velocities
    norms = np.linalg.norm(
        V,
        axis=2,
        keepdims=True
    )

    norms[norms == 0] = 1.0

    V_unit = V / norms

    mean_V = np.mean(V_unit, axis=0)

    mean_norms = np.linalg.norm(
        mean_V,
        axis=1
    )

    valid = mean_norms > 1e-10

    mean_unit = np.zeros_like(mean_V)

    mean_unit[valid] = (
        mean_V[valid] /
        mean_norms[valid, None]
    )

   #angles
    cos_theta = np.sum(
        V_unit * mean_unit[None, :, :],
        axis=2
    )

    cos_theta = np.clip(
        cos_theta,
        -1.0,
        1.0
    )

    angles = np.degrees(
        np.arccos(cos_theta)
    )

    angular_rmsd_per_atom = np.std(
        angles,
        axis=0
    )

    radii_per_atom = np.mean(
        R,
        axis=0
    )

  #store each atom seperately
    for i in range(len(sulfur_idx)):

        label = (
            f"{sulfur_info['resnames'][i]}"
            f"{sulfur_info['resids'][i]}"
            f"_{i+1}"
        )

        if label not in atom_accumulator:

            atom_accumulator[label] = {

                "label": label,

                "angular_rmsd_values": [],
                "radius_values": []
            }

        atom_accumulator[label][
            "angular_rmsd_values"
        ].append(
            angular_rmsd_per_atom[i]
        )

        atom_accumulator[label][
            "radius_values"
        ].append(
            radii_per_atom[i]
        )


#compute mean values
results = {}

for label, data in atom_accumulator.items():

    results[label] = {

        "mean_angular_rmsd": np.mean(
            data["angular_rmsd_values"]
        ),

        "mean_radius": np.mean(
            data["radius_values"]
        )
    }


#plotting
fig, ax = plt.subplots(figsize=(9, 7))

for label, data in sorted(results.items()):

    x = data["mean_radius"]
    y = data["mean_angular_rmsd"]

    color = "yellow"

    ax.scatter(
        x,
        y,
        s=220,
        color=color,
        edgecolors="black",
        linewidth=1.3,
        alpha=0.85
    )

    ax.annotate(
        label,
        (x, y),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=10
    )


ax.set_xlabel(
    "Radius from protein center (Å)",
    fontsize=15
)

ax.set_ylabel(
    "Angular RMSD (°)",
    fontsize=15
)

ax.set_title(
    "Homogeneous sulfur variability",
    fontsize=17
)

ax.tick_params(labelsize=12)

ax.grid(alpha=0.3)

plt.tight_layout()

plt.show()